# Notebook 11 ? Olah CSV Berukuran Besar

Baca dan salin CSV sedikit demi sedikit agar penggunaan memori lebih ringan. Bisa memilih beberapa kolom dan menyaring satu nilai. Proses ini tidak menyimpan seluruh file di memori sekaligus.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI
INPUT_PATH = Path("Input/input_besar.csv")
SEP = ","
CHUNK_SIZE = 100_000
USE_COLUMNS = None        # Contoh: ["id", "tanggal", "nilai"] atau None untuk semua kolom
FILTER_COLUMN = None      # Contoh: "status"; None berarti tanpa filter
FILTER_VALUE = None       # Contoh: "aktif"
OUTPUT_PATH = Path("Output/hasil_besar.csv")
OVERWRITE_OUTPUT = False  # Ubah ke True jika memang ingin mengganti hasil lama
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Proses per bagian

Sel ini menulis hasil baru. Jangan gunakan `OUTPUT_PATH` yang sama dengan file sumber.

In [ ]:
source = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
output = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
if source.resolve() == output.resolve(): raise ValueError("Pilih OUTPUT_PATH yang berbeda agar file sumber tidak tertimpa.")
if output.exists() and not OVERWRITE_OUTPUT: raise FileExistsError(f"File hasil sudah ada: {output}. Ubah OUTPUT_PATH atau set OVERWRITE_OUTPUT = True.")
if FILTER_COLUMN and USE_COLUMNS is not None and FILTER_COLUMN not in USE_COLUMNS:
    USE_COLUMNS = [*USE_COLUMNS, FILTER_COLUMN]
output.parent.mkdir(parents=True, exist_ok=True)
if output.exists(): output.unlink()
total_read = total_written = 0
for chunk_number, chunk in enumerate(pd.read_csv(source, sep=SEP, dtype="string", encoding="utf-8-sig", usecols=USE_COLUMNS, chunksize=CHUNK_SIZE), start=1):
    total_read += len(chunk)
    if FILTER_COLUMN:
        if FILTER_COLUMN not in chunk: raise KeyError(f"Kolom filter tidak ditemukan: {FILTER_COLUMN}")
        chunk = chunk.loc[chunk[FILTER_COLUMN].eq(FILTER_VALUE)]
    chunk.to_csv(output, mode="w" if chunk_number == 1 else "a", header=(chunk_number == 1), index=False, encoding=("utf-8-sig" if chunk_number == 1 else "utf-8"))
    total_written += len(chunk)
    if chunk_number % 10 == 0: print(f"Bagian {chunk_number}: dibaca {total_read:,}, ditulis {total_written:,} baris")
if total_read == 0:
    # Buat file kosong dengan header bila CSV sumber tidak memiliki baris data.
    header = pd.read_csv(source, sep=SEP, nrows=0, encoding="utf-8-sig", usecols=USE_COLUMNS)
    header.to_csv(output, index=False, encoding="utf-8-sig")
print(f"Selesai. Dibaca {total_read:,}, ditulis {total_written:,} baris -> {output.resolve()}")
